# Avaliação Comparativa das Políticas no Blackjack (com Contagem Hi-Lo)

## Objetivo
Este notebook consolida e compara os resultados obtidos por três estratégias no jogo Blackjack num ambiente realista de casino (baralho contínuo):

- **Política aleatória**;
- **Política básica**;
- **Política aprendida com Q-Learning (com contagem de cartas Hi-Lo)**.

O objetivo é produzir uma avaliação final para verificar se a introdução da 'temperatura' da mesa (True Count) permitiu ao agente de Q-Learning superar a vantagem matemática da banca.

## Papel desta etapa no projeto

Após a implementação do ambiente, a simulação das políticas de referência e o treino do agente com Q-Learning, esta etapa reúne os resultados gerados anteriormente para permitir uma comparação final entre as estratégias.

In [1]:
import pandas as pd
from utils_io import salvar_df, carregar_df

## Carregamento das bases

Vamos carregar a base de referência (políticas aleatória e básica) e a nova base do Q-Learning treinado com o sistema Hi-Lo.

In [2]:
# A base simples (criada com baralho contínuo)
df_politicas_base = carregar_df("blackjack_episodios_etapa_basica", pasta="dados")

# A base do agente inteligente com contagem de cartas
df_qlearning = carregar_df("blackjack_resultados_qlearning_hilo", pasta="dados")

Lendo arquivo: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_episodios_etapa_basica.csv
Lendo arquivo: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_qlearning_hilo.csv


## Padronização da estrutura dos dados

Vamos garantir que as tabelas possuem as mesmas colunas. Destaca-se a inclusão da coluna `true_count_final`, que permitirá análises aprofundadas sobre o comportamento do agente em mesas 'quentes' e 'frias'.

In [3]:
colunas_padrao = [
    "id_episodio",
    "politica",
    "resultado",
    "recompensa",
    "total_jogador",
    "total_dealer",
    "dealer_carta_visivel",
    "true_count_final",
    "qtd_passos",
    "mao_jogador",
    "mao_dealer"
]

df_politicas_base = df_politicas_base[colunas_padrao].copy()
df_qlearning = df_qlearning[colunas_padrao].copy()

## Consolidação dos resultados finais

Filtramos apenas as políticas de interesse e juntamos tudo num único DataFrame.

In [4]:
df_politicas_base = df_politicas_base[
    df_politicas_base["politica"].isin(["aleatoria", "basica"])
].copy()

df_resultados_finais = pd.concat(
    [df_politicas_base, df_qlearning],
    ignore_index=True
)

df_resultados_finais.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,true_count_final,qtd_passos,mao_jogador,mao_dealer
0,0,aleatoria,derrota,-1,9,21,10,0,1,"[5, 4]","[10, 1]"
1,1,aleatoria,vitoria,1,20,23,10,-1,1,"[10, 10]","[10, 3, 10]"
2,2,aleatoria,derrota,-1,27,15,5,-1,3,"[1, 3, 9, 4, 10]","[5, 10]"
3,3,aleatoria,vitoria,1,19,18,10,-1,2,"[10, 6, 3]","[10, 8]"
4,4,aleatoria,derrota,-1,26,14,8,0,3,"[7, 4, 4, 1, 10]","[8, 6]"


## Distribuição dos resultados por política

Cálculo do volume de episódios e das taxas de vitória, derrota e empate por cada estratégia.

In [5]:
comparacao_contagem = (
    df_resultados_finais
    .groupby(["politica", "resultado"])
    .size()
    .reset_index(name="qtd")
)

total_por_politica = (
    df_resultados_finais
    .groupby("politica")
    .size()
    .reset_index(name="total_episodios")
)

df_comparacao_final = comparacao_contagem.merge(
    total_por_politica,
    on="politica",
    how="left"
)

df_comparacao_final["taxa_resultado"] = (
    df_comparacao_final["qtd"] / df_comparacao_final["total_episodios"]
)

df_comparacao_final = df_comparacao_final.sort_values(
    by=["politica", "resultado"]
).reset_index(drop=True)

df_comparacao_final

,politica,resultado,qtd,total_episodios,taxa_resultado
0,aleatoria,derrota,64003,100000,0.64003
1,aleatoria,empate,4573,100000,0.04573
2,aleatoria,vitoria,31424,100000,0.31424
3,basica,derrota,48733,100000,0.48733
4,basica,empate,10453,100000,0.10453
5,basica,vitoria,40814,100000,0.40814
6,qlearning,derrota,48092,100000,0.48092
7,qlearning,empate,9366,100000,0.09366
8,qlearning,vitoria,42542,100000,0.42542


## Métricas agregadas por política

Consolidação de indicadores importantes como a recompensa média e a quantidade de passos tomados por cada política.

In [6]:
df_avaliacao_final = (
    df_resultados_finais
    .groupby("politica")
    .agg(
        episodios=("id_episodio", "count"),
        recompensa_media=("recompensa", "mean"),
        recompensa_total=("recompensa", "sum"),
        total_medio_jogador=("total_jogador", "mean"),
        total_medio_dealer=("total_dealer", "mean"),
        passos_medios=("qtd_passos", "mean")
    )
    .reset_index()
)

resumo_resultados = (
    df_resultados_finais
    .pivot_table(
        index="politica",
        columns="resultado",
        values="id_episodio",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

resumo_resultados.columns.name = None

for col in ["vitoria", "derrota", "empate"]:
    if col not in resumo_resultados.columns:
        resumo_resultados[col] = 0

df_avaliacao_final = df_avaliacao_final.merge(
    resumo_resultados[["politica", "vitoria", "derrota", "empate"]],
    on="politica",
    how="left"
)

df_avaliacao_final["taxa_vitoria"] = df_avaliacao_final["vitoria"] / df_avaliacao_final["episodios"]
df_avaliacao_final["taxa_derrota"] = df_avaliacao_final["derrota"] / df_avaliacao_final["episodios"]
df_avaliacao_final["taxa_empate"] = df_avaliacao_final["empate"] / df_avaliacao_final["episodios"]

## Taxas de estouro (Bust Rate)

Indicador do perfil de risco de cada política.

In [7]:
taxas_estouro = (
    df_resultados_finais
    .assign(
        estouro_jogador=lambda x: (x["total_jogador"] > 21).astype(int),
        estouro_dealer=lambda x: (x["total_dealer"] > 21).astype(int)
    )
    .groupby("politica")
    .agg(
        taxa_estouro_jogador=("estouro_jogador", "mean"),
        taxa_estouro_dealer=("estouro_dealer", "mean")
    )
    .reset_index()
)

df_avaliacao_final = df_avaliacao_final.merge(
    taxas_estouro,
    on="politica",
    how="left"
)

df_avaliacao_final = df_avaliacao_final.sort_values(
    by=["taxa_vitoria", "recompensa_media"],
    ascending=False
).reset_index(drop=True)

df_avaliacao_final

,politica,episodios,recompensa_media,recompensa_total,total_medio_jogador,total_medio_dealer,passos_medios,vitoria,derrota,empate,taxa_vitoria,taxa_derrota,taxa_empate,taxa_estouro_jogador,taxa_estouro_dealer
0,qlearning,100000,-0.05550,-5550,19.07966,19.42846,1.57705,42542,48092,9366,0.42542,0.48092,0.09366,0.18813,0.23572
1,basica,100000,-0.07919,-7919,20.31796,18.69501,1.62601,40814,48733,10453,0.40814,0.48733,0.10453,0.28110,0.20302
2,aleatoria,100000,-0.32579,-32579,18.30806,18.69384,1.34464,31424,64003,4573,0.31424,0.64003,0.04573,0.28082,0.20287


## Síntese Comparativa

Visão focada nas métricas essenciais que determinarão o sucesso do Agente de Q-Learning.

In [8]:
df_avaliacao_final[[
    "politica",
    "episodios",
    "taxa_vitoria",
    "taxa_derrota",
    "taxa_empate",
    "recompensa_media",
    "taxa_estouro_jogador"
]]

,politica,episodios,taxa_vitoria,taxa_derrota,taxa_empate,recompensa_media,taxa_estouro_jogador
0,qlearning,100000,0.42542,0.48092,0.09366,-0.05550,0.18813
1,basica,100000,0.40814,0.48733,0.10453,-0.07919,0.28110
2,aleatoria,100000,0.31424,0.64003,0.04573,-0.32579,0.28082


## Persistência dos resultados finais

Gravamos as tabelas finais identificando-as com o sufixo `_hilo` para podermos utilizá-las na Análise Exploratória sem subscrever versões anteriores.

In [9]:
salvar_df(df_comparacao_final, "blackjack_comparacao_final_hilo", pasta="resultados")
salvar_df(df_avaliacao_final, "blackjack_avaliacao_final_hilo", pasta="resultados")
salvar_df(df_resultados_finais, "blackjack_resultados_completos_hilo", pasta="dados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_comparacao_final_hilo.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_avaliacao_final_hilo.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_resultados_completos_hilo.csv


## Conclusão

Este caderno consolida a avaliação final do projeto.

A análise comparativa permite verificar o comportamento da política aprendida (que agora domina a contagem de cartas) contra as abordagens tradicionais.
O próximo e último passo do fluxo de dados consiste em importar a tabela `blackjack_resultados_completos_hilo` para o caderno de **Análise Exploratória**, onde as correlações entre vitórias e o *True Count* poderão ser demonstradas visualmente.